# Linear Regression From Scratch

Implementing linear regression's math directly instead of calling `sklearn.linear_model.LinearRegression`, to actually understand what `.fit()` is doing under the hood.

**Two versions here:**
1. **Scalar closed-form** — single feature, derived from minimizing sum of squared errors by hand
2. **Matrix / Normal Equation** — generalizes to any number of features using linear algebra

Both are compared against sklearn's implementation to confirm they produce the same result.

## 0. Imports

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.datasets import load_diabetes
from sklearn.metrics import r2_score

## 1. Scalar Closed-Form (Single Feature)

For one feature, the OLS solution for slope `m` and intercept `b` has a direct formula, derived by taking the derivative of the squared-error loss and setting it to zero:

$$m = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2} \qquad b = \bar{y} - m\bar{x}$$

Implementing this directly (looping over rows) makes the formula concrete before jumping to the vectorized matrix version.

In [2]:
class MeraLR:

    def __init__(self):
        self.m = None
        self.b = None

    def fit(self, X_train, y_train):
        num = 0
        den = 0

        for i in range(X_train.shape[0]):
            num = num + ((X_train[i] - X_train.mean()) * (y_train[i] - y_train.mean()))
            den = den + ((X_train[i] - X_train.mean()) * (X_train[i] - X_train.mean()))

        self.m = num / den
        self.b = y_train.mean() - (self.m * X_train.mean())

    def predict(self, X_test):
        return self.m * X_test + self.b

### Try it on `placement.csv`

In [3]:
df = pd.read_csv('placement.csv')
df.head()

,cgpa,package
0,6.48,2.37
1,6.84,2.79
2,4.76,1.72
3,8.54,4.32
4,5.11,2.22


In [4]:
X = df.iloc[:,0].values
y = df.iloc[:,1].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
X_train.shape

(160,)

In [5]:
lr = MeraLR()
lr.fit(X_train, y_train)
print(f"m: {lr.m:.4f}")
print(f"b: {lr.b:.4f}")

m: 0.5338
b: -0.7170


In [6]:
lr.predict(X_test[0])

np.float64(2.48556257173599)

In [7]:
# Compare against sklearn's LinearRegression on the same split
sk_lr = LinearRegression()
sk_lr.fit(X_train.reshape(-1,1), y_train)
print(f"sklearn m: {sk_lr.coef_[0]:.4f}")
print(f"sklearn b: {sk_lr.intercept_:.4f}")

sklearn m: 0.5338
sklearn b: -0.7170


Matches sklearn's output — confirming the manual scalar formula is correct for the single-feature case.

## 2. Matrix / Normal Equation (Any Number of Features)

The scalar formula above only works for one feature. The general solution for any number of features uses the **Normal Equation**:

$$\boldsymbol{\beta} = (X^T X)^{-1} X^T y$$

where `X` has a column of 1s prepended (so the first entry of `β` becomes the intercept, and the rest become the coefficients). This is the same OLS objective, just solved with linear algebra instead of a per-feature formula.

In [8]:
X, y = load_diabetes(return_X_y=True)
X.shape, y.shape

((442, 10), (442,))

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
print(X_train.shape)
print(X_test.shape)

(353, 10)
(89, 10)


### Baseline: sklearn's LinearRegression

In [10]:
reg = LinearRegression()
reg.fit(X_train, y_train)

y_pred = reg.predict(X_test)
r2_score(y_test, y_pred)

0.439933866156897

In [11]:
reg.coef_, reg.intercept_

(array([  -9.15865318, -205.45432163,  516.69374454,  340.61999905,
        -895.5520019 ,  561.22067904,  153.89310954,  126.73139688,
         861.12700152,   52.42112238]),
 np.float64(151.88331005254167))

### Implementing the Normal Equation Ourselves

In [12]:
class MeraLR:

    def __init__(self):
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X_train, y_train):
        # prepend a column of 1s so the intercept is solved for as part of beta
        X_train = np.insert(X_train, 0, 1, axis=1)

        betas = np.linalg.inv(np.dot(X_train.T, X_train)).dot(X_train.T).dot(y_train)
        self.intercept_ = betas[0]
        self.coef_ = betas[1:]

    def predict(self, X_test):
        return np.dot(X_test, self.coef_) + self.intercept_

In [13]:
lr = MeraLR()
lr.fit(X_train, y_train)

In [14]:
y_pred = lr.predict(X_test)
r2_score(y_test, y_pred)

0.4399338661568968

In [15]:
lr.coef_, lr.intercept_

(array([  -9.15865318, -205.45432163,  516.69374454,  340.61999905,
        -895.5520019 ,  561.22067904,  153.89310954,  126.73139688,
         861.12700152,   52.42112238]),
 np.float64(151.88331005254176))

Same R², same coefficients, same intercept as sklearn's `LinearRegression` — confirming `.fit()` is, at its core, solving exactly this normal equation (sklearn actually uses a more numerically stable method like SVD internally, but the result is equivalent).

---
## Regression From Scratch Cheat Sheet

Quick reference for the patterns shown above.

### Scalar Closed-Form (1 feature)
| Formula | Description |
|---|---|
| `m = Σ(x-x̄)(y-ȳ) / Σ(x-x̄)²` | Slope |
| `b = ȳ - m·x̄` | Intercept |

### Matrix / Normal Equation (any # of features)
| Code | Description |
|---|---|
| `np.insert(X, 0, 1, axis=1)` | Prepend a column of 1s so the intercept is solved as part of `β` |
| `np.linalg.inv(X.T @ X) @ X.T @ y` | The Normal Equation — solves for all coefficients + intercept at once |
| `betas[0]` | Intercept |
| `betas[1:]` | Coefficients |

### Gotchas
- The Normal Equation requires `X.T @ X` to be invertible — fails or becomes unstable with highly correlated (collinear) features
- `np.linalg.inv` is fine for learning, but sklearn uses SVD-based solvers internally for better numerical stability at scale
- Always sanity-check from-scratch implementations against sklearn's output on the same data/split